In [1]:
import pandas as pd

df = pd.read_csv('train.csv')
df.head()

RuntimeError: module was compiled against NumPy C-API version 0x10 (NumPy 1.23) but the running NumPy has C-API version 0xf. Check the section C-API incompatibility at the Troubleshooting ImportError section at https://numpy.org/devdocs/user/troubleshooting-importerror.html#c-api-incompatibility for indications on how to solve this problem.

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [3]:
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [4]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [5]:
print("Процент выживаемости пассажиров первого, второго и третьего классов относительно всего экипажа")
df.groupby(["Pclass"])["Survived"].sum() / df.shape[0] * 100

Процент выживаемости пассажиров первого, второго и третьего классов относительно всего экипажа


Pclass
1    15.263749
2     9.764310
3    13.355780
Name: Survived, dtype: float64

In [6]:
print("Процент выживаемости пассажиров в каждом классе")
df.groupby(["Pclass"])["Survived"].sum() / df.groupby(["Pclass"])['PassengerId'].count() * 100

Процент выживаемости пассажиров в каждом классе


Pclass
1    62.962963
2    47.282609
3    24.236253
dtype: float64

In [7]:
male_df = df[df['Sex'] == 'male'].copy()
male_df[['Last name', 'First name']] = male_df['Name'].str.split(',', expand=True)
male_df["First name"] = male_df["First name"].replace(to_replace=r".+?\. ", value="", regex=True)
male_df["First name"] = male_df["First name"].replace(to_replace=r' .+?$', value="", regex=True) # я решила оставить первое имя из сложных имён
male_names_total = male_df.groupby(["First name"])["PassengerId"].count()
male_names_total = male_names_total.sort_values(ascending=False)

In [8]:
female_df = df[df['Sex'] == 'female'].copy()
female_df["First name"] = female_df["Name"]
female_df.loc[female_df['Name'].str.contains('Mrs.', na=False), 'First name'] = female_df['First name'].str.replace("\)", "", regex=True)
female_df.loc[female_df['Name'].str.contains('Mrs.', na=False), 'First name'] = female_df['First name'].str.replace(r".+?\(", "", regex=True)
female_df.loc[female_df['Name'].str.contains('Mrs.', na=False), 'First name'] = female_df['First name'].str.replace(r"\"", "", regex=True)
female_df.loc[~female_df['Name'].str.contains('Mrs.', na=False), 'First name'] = female_df['First name'].str.replace(".+?\. ", "", regex=True)
female_df['First name'] = female_df['First name'].str.replace(" .+?$", "", regex=True) # оставляю только первое слово в имени
female_names_total = female_df.groupby(["First name"])["PassengerId"].count()
female_names_total = female_names_total.sort_values(ascending=False)

In [9]:
print(f"Самое популярное мужское имя: {male_names_total.index[0]}")
print(f"Самое популярное женское имя: {female_names_total.index[0]}")

Самое популярное мужское имя: William
Самое популярное женское имя: Anna


In [10]:
male_names_class = male_df.groupby(["Pclass", "First name"])["First name"].count()
male_names_class = male_df.groupby(["Pclass", "First name"])["PassengerId"].count().to_frame()
for i in range(1,4):
    print(f'Самое популярное мужское имя в {i} классе: {male_names_class["PassengerId"].loc[i].idxmax()}')
    print(f'Частота: {male_names_class["PassengerId"].loc[i].max()}')

Самое популярное мужское имя в 1 классе: William
Частота: 11
Самое популярное мужское имя в 2 классе: William
Частота: 9
Самое популярное мужское имя в 3 классе: William
Частота: 15


In [11]:
female_names_class = female_df.groupby(["Pclass", "First name"])["PassengerId"].count().to_frame()
for i in range(1,4):
    print(f'Самое популярное женское имя в {i} классе: {female_names_class["PassengerId"].loc[i].idxmax()}')
    print(f'Частота: {female_names_class["PassengerId"].loc[i].max()}')

Самое популярное женское имя в 1 классе: Elizabeth
Частота: 5
Самое популярное женское имя в 2 классе: Elizabeth
Частота: 5
Самое популярное женское имя в 3 классе: Anna
Частота: 9


In [12]:
print("Пассажиры старше 44 лет")
df[df['Age'] > 44].sample(5, random_state=1)

Пассажиры старше 44 лет


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
712,713,1,1,"Taylor, Mr. Elmer Zebley",male,48.0,1,0,19996,52.0000,C126,S
406,407,0,3,"Widegren, Mr. Carl/Charles Peter",male,51.0,0,0,347064,7.7500,NaN,S
482,483,0,3,"Rouse, Mr. Richard Henry",male,50.0,0,0,A/5 3594,8.0500,NaN,S
645,646,1,1,"Harper, Mr. Henry Sleeper",male,48.0,1,0,PC 17572,76.7292,D33,C
556,557,1,1,"Duff Gordon, Lady. (Lucille Christiana Sutherl...",female,48.0,1,0,11755,39.6000,A16,C


In [13]:
print('Пассажиры мужского пола младше 44 лет')
df[(df['Age'] < 44) & (df['Sex'] == 'male')].sample(5, random_state=1)

Пассажиры мужского пола младше 44 лет


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
752,753,0,3,"Vande Velde, Mr. Johannes Joseph",male,33.0,0,0,345780,9.500,NaN,S
286,287,1,3,"de Mulder, Mr. Theodore",male,30.0,0,0,345774,9.500,NaN,S
579,580,1,3,"Jussila, Mr. Eiriik",male,32.0,0,0,STON/O 2. 3101286,7.925,NaN,S
664,665,1,3,"Lindqvist, Mr. Eino William",male,20.0,1,0,STON/O 2. 3101285,7.925,NaN,S
234,235,0,2,"Leyson, Mr. Robert William Norman",male,24.0,0,0,C.A. 29566,10.500,NaN,S


In [14]:
df_cabins = df.groupby(['Cabin'])['PassengerId'].count().reset_index()
df_cabins = df_cabins.rename(columns={'PassengerId': 'n'})
print('Количество n-местных кают (n > 1)')
df_cabins[df_cabins['n'] > 1].groupby(['n']).count()

Количество n-местных кают (n > 1)


,Cabin
n,
2,38
3,5
4,3
